## Comparison Summary (Current Outputs)

| # | Table | Schema Matches | Full Match | Synapse Rows | Databricks Rows | Row Count Matches | Matched Rows | Match % of Synapse |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW | True | False | 1,429,931 | 1,429,946 | False | 1,429,916 | 99.99895% |
| 2 | VW_HSTG_UDLGENSD03_DBO_GEN_CODEIPT_DEF | False | True | 7 | 7 | True | 7 | 100.00000% |
| 3 | VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT | False | False | 1,429,931 | 1,429,946 | False | 1,429,931 | 100.00000% |
| 4 | VW_HSTG_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF | False | True | 15 | 15 | True | 15 | 100.00000% |
| 5 | VW_HSTG_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF | False | True | 31 | 31 | True | 31 | 100.00000% |
| 6 | VW_HSTG_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF | False | True | 88 | 88 | True | 88 | 100.00000% |

# Change Data Feed (CDF) Guide
## Schema: `lakehouse.analytics_and_enablement_staging_pi_systems`

---

## Architecture Overview

Tables in this schema follow a **two-catalog pattern** managed by the RTLH (Rio Tinto Lakehouse) framework:

| Catalog | Purpose | Table Type |
|---------|---------|------------|
| `lakehouse` | Current state (latest snapshot) | VIEW over `_lakehouse_aws_syd` |
| `lakehouse_history` | Change history (CDF) | VIEW over `_lakehouse_aws_syd_history` |

Both catalogs expose **views** that wrap the physical Delta tables in internal catalogs (`_lakehouse_aws_syd` and `_lakehouse_aws_syd_history`). Users never access the internal catalogs directly.

---

## Tables and Their CDF Counterparts

| Current State Table (`lakehouse.*`) | CDF Table (`lakehouse_history.*`) |
|--------------------------------------|-------------------------------------|
| `opsdata_mis_rtit_headgrade` | `opsdata_mis_rtit_headgrade_cdf` |
| `opsdata_mis_rtit_sandmetrics` | `opsdata_mis_rtit_sandmetrics_cdf` |
| `opsdata_pm_reporting_data` | `opsdata_pm_reporting_data_cdf` |
| `opsdata_pm_reporting_recovery_data` | `opsdata_pm_reporting_recovery_data_cdf` |
| `globalpi_pm_reporting_locations_hierarchy` | `globalpi_pm_reporting_locations_hierarchy_cdf` |

Every table in `lakehouse.analytics_and_enablement_staging_pi_systems` has a 1:1 CDF companion in `lakehouse_history.analytics_and_enablement_staging_pi_systems`.

---

## Column Structure

### Current State Tables (`lakehouse.*`)
- Business columns (e.g. `actual`, `budget`, `datefrom`, `dateto`, `sitecode`, etc.)
- `rtlh_ingestion_time` — timestamp when the row was ingested
- `rtlh_row_hash` — hash of the row used for change detection

### CDF Tables (`lakehouse_history.*`)
Same business columns as above, plus:
- `_change_type` — Delta CDF metadata: `insert`, `update_preimage`, `update_postimage`, or `delete`
- `_commit_timestamp` — when the change was committed to the Delta table
- `_commit_version` — the Delta table version number of the commit
- `rtlh_cdc_operation` — RTLH-specific CDC operation marker
- `rtlh_ingestion_time` — ingestion timestamp
- `rtlh_job_run_id` — the job run that produced this change
- `rtlh_row_hash` — row hash for deduplication

---

## How CDF Works

1. **Ingestion**: The RTLH framework ingests data into the physical Delta tables in `_lakehouse_aws_syd`. It uses `rtlh_row_hash` to detect changes (inserts, updates, deletes) compared to the previous state.

2. **Delta CDF**: The underlying Delta tables have `delta.enableChangeDataFeed = true`. Every write operation automatically records row-level changes with metadata (`_change_type`, `_commit_timestamp`, `_commit_version`).

3. **CDF Views**: The `lakehouse_history` catalog exposes these changes via views that read from the internal `_lakehouse_aws_syd_history` tables, making the change feed accessible without direct access to internal catalogs.

---

## Querying CDF

### Get recent changes (last 24 hours)
```sql
SELECT *
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE _commit_timestamp > current_timestamp() - INTERVAL 1 DAY
ORDER BY _commit_timestamp DESC
```

### Get changes by operation type
```sql
SELECT _change_type, COUNT(*) as change_count
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE _commit_timestamp > '2026-06-01'
GROUP BY _change_type
```

### Track a specific record's history
```sql
SELECT *
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE tagname = 'your_tag_name'
ORDER BY _commit_timestamp
```

---

## Key Differences from `_ogr_azr_syd` Tables

Tables in `_ogr_azr_syd` (e.g. `gensd03_sicom` schema) are **direct MANAGED Delta tables** — not views. They:
- Have the same `rtlh_row_hash` and `rtlh_ingestion_time` columns
- Do NOT have pre-built CDF companion views
- Require using `table_changes()` function directly (if CDF is enabled)
- Live in a raw/internal catalog without the `lakehouse`/`lakehouse_history` facade

---

## Summary

The `lakehouse` pattern provides a clean separation:
- **Read current data** → query `lakehouse.analytics_and_enablement_staging_pi_systems.<table>`
- **Read change history** → query `lakehouse_history.analytics_and_enablement_staging_pi_systems.<table>_cdf`

No need to enable CDF manually or use `table_changes()` — the RTLH framework handles this automatically and exposes it via the `lakehouse_history` catalog.

In [1]:
from databricks import sql
import polars as pl
import pyodbc

# Synapse Connection
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = ''
synapse_driver = '{ODBC Driver 17 for SQL Server}'
synapse_schema_name = 'HSTG_V'
synapse_table_name = 'VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""
synapse_conn = pyodbc.connect(synapse_connection_string)

# Databricks Connection
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token
)

In [2]:
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [8]:
def compare_synapse_vs_databricks(
    databricks_conn,
    synapse_conn,
    synapse_schema_name,
    synapse_table_name,
    databricks_catalog_name,
    databricks_schema_name,
    databricks_table_name,
):
    # Type normalization helpers
    def base_type(dtype: str) -> str:
        return str(dtype).split("(")[0].strip().upper() if dtype is not None else None

    databricks_to_synapse_type_map = {
        "TINYINT": "TINYINT",
        "SMALLINT": "SMALLINT",
        "INT": "INT",
        "INTEGER": "INT",
        "BIGINT": "BIGINT",
        "FLOAT": "REAL",
        "REAL": "REAL",
        "DOUBLE": "FLOAT",
        "DECIMAL": "DECIMAL",
        "NUMERIC": "NUMERIC",
        "STRING": "VARCHAR",
        "VARCHAR": "VARCHAR",
        "CHAR": "CHAR",
        "BINARY": "VARBINARY",
        "DATE": "DATE",
        "TIMESTAMP": "DATETIME2",
        "TIMESTAMP_NTZ": "DATETIME2",
        "TIMESTAMP_LTZ": "DATETIMEOFFSET",
        "BOOLEAN": "BIT",
    }

    # 1) Read schema metadata
    synapse_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as synapse_data_type
        from information_schema.columns
        where table_schema = '{synapse_schema_name}'
          and table_name = '{synapse_table_name}'
          and column_name not like 'OMD%'
          and column_name not like 'HSTG%'
        order by ordinal_position
    """

    databricks_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as databricks_data_type
        from {databricks_catalog_name}.information_schema.columns
        where table_schema = '{databricks_schema_name}'
          and table_name = '{databricks_table_name}'
          and column_name not like 'rtlh%'
    """

    synapse_schema_df = pl.read_database(synapse_schema_query, synapse_conn)
    databricks_schema_df = pl.read_database(databricks_schema_query, databricks_conn)

    # Map Databricks type -> expected Synapse type
    databricks_schema_df = databricks_schema_df.with_columns(
        pl.col("databricks_data_type")
        .map_elements(lambda x: databricks_to_synapse_type_map.get(base_type(x), None), return_dtype=pl.Utf8)
        .alias("synapse_expected_type")
    )

    # 2) Compare schema
    schema_comparison_df = synapse_schema_df.join(
        databricks_schema_df,
        on="column_name",
        how="full"
    )

    schema_mismatches_df = schema_comparison_df.filter(
        pl.col("synapse_data_type").is_null()
        | pl.col("databricks_data_type").is_null()
        | (
            pl.col("synapse_data_type").map_elements(base_type, return_dtype=pl.Utf8)
            != pl.col("synapse_expected_type").map_elements(base_type, return_dtype=pl.Utf8)
        )
    )

    schema_matches = schema_mismatches_df.height == 0

    # 3) Row count check (same notebook logic)
    synapse_count_query = f"""
        select count(*) as row_count
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_count_query = f"""
        select count(*) as row_count
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    synapse_row_count = int(pl.read_database(synapse_count_query, synapse_conn)["row_count"][0])
    databricks_row_count = int(pl.read_database(databricks_count_query, databricks_conn)["row_count"][0])
    row_count_matches = synapse_row_count == databricks_row_count

    # 4) Full match on common columns with detailed alignment and normalization
    common_cols = sorted(
        set(synapse_schema_df["column_name"].to_list())
        & set(databricks_schema_df["column_name"].to_list())
    )

    if not common_cols:
        return {
            "schema_matches": schema_matches,
            "row_count_matches": row_count_matches,
            "full_match": False,
            "reason": "No common columns found for full comparison.",
            "synapse_row_count": synapse_row_count,
            "databricks_row_count": databricks_row_count,
            "schema_mismatches_df": schema_mismatches_df,
            "schema_comparison_df": schema_comparison_df,
            "matched_rows": 0,
            "pct_of_synapse": 0.0,
            "pct_of_databricks": 0.0,
            "pct_overall": 0.0,
        }

    synapse_cols_sql = ", ".join([f"[{c}] as [SYNAPSE_{c}]" for c in common_cols])
    databricks_cols_sql = ", ".join([f"`{c}` as `DATABRICKS_{c}`" for c in common_cols])

    synapse_full_query = f"""
        select {synapse_cols_sql}
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_full_query = f"""
        select {databricks_cols_sql}
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    synapse_full_table_df = pl.read_database(synapse_full_query, synapse_conn)
    databricks_full_table_df = pl.read_database(databricks_full_query, databricks_conn)

    # Normalize timezone-aware datetime columns in Databricks data
    tz_cols = [
        name for name, dtype in databricks_full_table_df.schema.items()
        if dtype == pl.Datetime and getattr(dtype, "time_zone", None) is not None
    ]
    if tz_cols:
        databricks_full_table_df = databricks_full_table_df.with_columns(
            [pl.col(c).dt.replace_time_zone(None) for c in tz_cols]
        )

    # ============ Enhanced Matching Logic ============
    INT_LIKE = {
        pl.Boolean, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64
    }
    FLOAT_LIKE = {pl.Float32, pl.Float64}

    # 1) Align to the same logical column names
    pairs = [
        {"synapse_column_name": c, "databricks_column_name": c}
        for c in common_cols
    ]

    syn_aligned = synapse_full_table_df.select([
        pl.col(f"SYNAPSE_{r['synapse_column_name']}").alias(r["synapse_column_name"])
        for r in pairs
    ])

    dbx_aligned = databricks_full_table_df.select([
        pl.col(f"DATABRICKS_{r['databricks_column_name']}").alias(r["synapse_column_name"])
        for r in pairs
    ])

    # 2) Normalize dtypes (ints -> Int64, floats -> Float64, datetimes -> naive us)
    def normalize_types(df: pl.DataFrame) -> pl.DataFrame:
        exprs = []
        for c, d in df.schema.items():
            if d in INT_LIKE:
                exprs.append(pl.col(c).cast(pl.Int64).alias(c))
            elif d in FLOAT_LIKE:
                exprs.append(pl.col(c).cast(pl.Float64).alias(c))
            elif str(d).startswith("Datetime"):
                e = pl.col(c)
                if "time_zone" in str(d):
                    e = e.dt.replace_time_zone(None)
                exprs.append(e.dt.cast_time_unit("us").alias(c))
        return df.with_columns(exprs) if exprs else df

    syn_norm = normalize_types(syn_aligned)
    dbx_norm = normalize_types(dbx_aligned)

    # 3) Hash after normalization
    syn_hashed = syn_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))
    dbx_hashed = dbx_norm.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))

    full_match_exact = syn_hashed["row_hash"].sort().equals(dbx_hashed["row_hash"].sort())

    # Counts and detailed matching
    syn_counts = (
        syn_hashed
        .group_by("row_hash")
        .len()
        .rename({"len": "syn_cnt"})
    )

    dbx_counts = (
        dbx_hashed
        .group_by("row_hash")
        .len()
        .rename({"len": "dbx_cnt"})
    )

    cmp = (
        syn_counts.join(dbx_counts, on="row_hash", how="full")
        .with_columns(
            pl.col("syn_cnt").fill_null(0).cast(pl.Int64),
            pl.col("dbx_cnt").fill_null(0).cast(pl.Int64),
        )
        .with_columns(
            pl.min_horizontal("syn_cnt", "dbx_cnt").alias("matched_cnt")
        )
    )

    # Use sums from counts (guaranteed same universe as matched_cnt)
    syn_total = int(syn_counts["syn_cnt"].sum())
    dbx_total = int(dbx_counts["dbx_cnt"].sum())
    matched_rows = int(cmp["matched_cnt"].sum())

    # Defensive clamp (prevents >100% if anything odd leaks in)
    matched_rows = min(matched_rows, syn_total, dbx_total)

    pct_of_synapse = 100.0 * matched_rows / syn_total if syn_total else 100.0
    pct_of_databricks = 100.0 * matched_rows / dbx_total if dbx_total else 100.0
    pct_overall = 100.0 * matched_rows / max(syn_total, dbx_total) if max(syn_total, dbx_total) else 100.0

    return {
        "schema_matches": schema_matches,
        "row_count_matches": row_count_matches,
        "full_match": full_match_exact,
        "synapse_row_count": synapse_row_count,
        "databricks_row_count": databricks_row_count,
        "schema_mismatches_df": schema_mismatches_df,
        "schema_comparison_df": schema_comparison_df,
        "common_columns_compared": common_cols,
        "matched_rows": matched_rows,
        "syn_total": syn_total,
        "dbx_total": dbx_total,
        "pct_of_synapse": pct_of_synapse,
        "pct_of_databricks": pct_of_databricks,
        "pct_overall": pct_overall,
    }

In [5]:
# UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_suivietatequipementrownumber_view",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: True
Full match after schema check: False
Synapse row count: 1429931
Databricks row count: 1429946
Matched: 1429916/1429931 (99.99895%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""CAUSE""","""VARCHAR""","""CAUSE""","""STRING""","""VARCHAR"""
"""DATEDEBUT""","""DATETIME2""","""DATEDEBUT""","""TIMESTAMP""","""DATETIME2"""
"""DATEFIN""","""DATETIME2""","""DATEFIN""","""TIMESTAMP""","""DATETIME2"""
"""DESCRIPTION""","""VARCHAR""","""DESCRIPTION""","""STRING""","""VARCHAR"""
"""DUREESEC""","""INT""","""DUREESEC""","""INT""","""INT"""
"""IDCODEARRET""","""INT""","""IDCODEARRET""","""INT""","""INT"""
"""IDEMPLOYE""","""INT""","""IDEMPLOYE""","""INT""","""INT"""
"""IDEQUIPEMENT""","""INT""","""IDEQUIPEMENT""","""INT""","""INT"""
"""IDOPERATION""","""INT""","""IDOPERATION""","""INT""","""INT"""


In [6]:
# UDLGENSD03_DBO_GEN_CODEIPT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_GEN_CODEIPT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_gen_codeipt_def",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: False
Full match after schema check: True
Synapse row count: 7
Databricks row count: 7
Matched: 7/7 (100.00000%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""IDCODEIPT""","""INT""","""IDCODEIPT""","""INT""","""INT"""
"""ACTIF""","""INT""","""ACTIF""","""BOOLEAN""","""BIT"""
"""DESCRIPTION""","""VARCHAR""","""DESCRIPTION""","""STRING""","""VARCHAR"""
"""ENVOIREP""","""INT""","""ENVOIREP""","""BOOLEAN""","""BIT"""
"""ISDELETED""","""INT""","""ISDELETED""","""BOOLEAN""","""BIT"""
"""NOM""","""VARCHAR""","""NOM""","""STRING""","""VARCHAR"""
"""NOMPERMANENT""","""VARCHAR""","""NOMPERMANENT""","""STRING""","""VARCHAR"""
"""ORDREAFFICHAGE""","""INT""","""ORDREAFFICHAGE""","""INT""","""INT"""


In [7]:
# UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_suivietatequipement",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: False
Full match after schema check: False
Synapse row count: 1429931
Databricks row count: 1429946
Matched: 1429931/1429931 (100.00000%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""DATEDEBUT""","""DATETIME2""","""DATEDEBUT""","""TIMESTAMP""","""DATETIME2"""
"""IDEQUIPEMENT""","""INT""","""IDEQUIPEMENT""","""INT""","""INT"""
"""AUTOMATIQUE""","""INT""","""AUTOMATIQUE""","""BOOLEAN""","""BIT"""
"""CAUSE""","""VARCHAR""","""CAUSE""","""STRING""","""VARCHAR"""
"""DESCRIPTION""","""VARCHAR""","""DESCRIPTION""","""STRING""","""VARCHAR"""
"""IDCODEARRET""","""INT""","""IDCODEARRET""","""INT""","""INT"""
"""IDEMPLOYE""","""INT""","""IDEMPLOYE""","""INT""","""INT"""
"""IDOPERATION""","""INT""","""IDOPERATION""","""INT""","""INT"""
"""IDSOURCE""","""INT""","""IDSOURCE""","""INT""","""INT"""


In [8]:
# UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_groupeequipement_def",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: False
Full match after schema check: True
Synapse row count: 15
Databricks row count: 15
Matched: 15/15 (100.00000%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""IDGROUPEEQUIPEMENT""","""INT""","""IDGROUPEEQUIPEMENT""","""INT""","""INT"""
"""ACTIF""","""INT""","""ACTIF""","""BOOLEAN""","""BIT"""
"""DESCRIPTION""","""VARCHAR""","""DESCRIPTION""","""STRING""","""VARCHAR"""
"""ENVOIREP""","""INT""","""ENVOIREP""","""BOOLEAN""","""BIT"""
"""IDFAMILLEEQUIPEMENT""","""INT""","""IDFAMILLEEQUIPEMENT""","""INT""","""INT"""
"""IDUNITE""","""INT""","""IDUNITE""","""INT""","""INT"""
"""ISDELETED""","""INT""","""ISDELETED""","""BOOLEAN""","""BIT"""
"""NOM""","""VARCHAR""","""NOM""","""STRING""","""VARCHAR"""
"""NOMPERMANENT""","""VARCHAR""","""NOMPERMANENT""","""STRING""","""VARCHAR"""


In [9]:
# UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_modeleequipement_def",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: False
Full match after schema check: True
Synapse row count: 31
Databricks row count: 31
Matched: 31/31 (100.00000%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""IDMODELEEQUIPEMENT""","""INT""","""IDMODELEEQUIPEMENT""","""INT""","""INT"""
"""DESCRIPTION""","""VARCHAR""","""DESCRIPTION""","""STRING""","""VARCHAR"""
"""ENVOIREP""","""INT""","""ENVOIREP""","""BOOLEAN""","""BIT"""
"""IDGROUPEEQUIPEMENT""","""INT""","""IDGROUPEEQUIPEMENT""","""INT""","""INT"""
"""ISDELETED""","""INT""","""ISDELETED""","""BOOLEAN""","""BIT"""
"""NOM""","""VARCHAR""","""NOM""","""STRING""","""VARCHAR"""
"""NOMPERMANENT""","""VARCHAR""","""NOMPERMANENT""","""STRING""","""VARCHAR"""


In [10]:
# VW_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="HSTG_V",
    synapse_table_name="VW_HSTG_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF",
    databricks_catalog_name="_ogr_azr_syd",
    databricks_schema_name="gensd03_sicom",
    databricks_table_name="dbo_hsp_equipement_def",
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Full match after schema check: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")
result["schema_comparison_df"]

Schema Matches: False
Full match after schema check: True
Synapse row count: 88
Databricks row count: 88
Matched: 88/88 (100.00000%)


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""IDEQUIPEMENT""","""INT""","""IDEQUIPEMENT""","""INT""","""INT"""
"""ACTIF""","""INT""","""ACTIF""","""BOOLEAN""","""BIT"""
"""COMMENTAIRE""","""VARCHAR""","""COMMENTAIRE""","""STRING""","""VARCHAR"""
"""COMPILATIONDISPONIBILITE""","""INT""","""COMPILATIONDISPONIBILITE""","""BOOLEAN""","""BIT"""
"""DIAMETRETREPAN""","""FLOAT""","""DIAMETRETREPAN""","""DOUBLE""","""FLOAT"""
"""ENLOCATION""","""INT""","""ENLOCATION""","""BOOLEAN""","""BIT"""
"""ENVOIREP""","""INT""","""ENVOIREP""","""BOOLEAN""","""BIT"""
"""FACTEURAJUSTEMENT""","""FLOAT""","""FACTEURAJUSTEMENT""","""DOUBLE""","""FLOAT"""
"""FACTEURCONVERSION""","""FLOAT""","""FACTEURCONVERSION""","""DOUBLE""","""FLOAT"""


In [ ]:
from azure.identity import AzureCliCredential
from azure.keyvault.secrets import SecretClient
import json

vault_url = "https://RT-C096-WAS-NPE-AKV-002.vault.azure.net"

credential = AzureCliCredential(additionally_allowed_tenants=["*"])
client = SecretClient(vault_url=vault_url, credential=credential)

# 1. CREATE the connection info
connection_info = {
    "servertype": "SQL Server",
    "server": "AUBOYSQL6.corp.riotinto.org",
    "database": "ODS_AVEVA_PROD_MANAGEMENT",
    "authmethod": "SQL",
    "username": "ODP-ETL-Databricks",
    "password": "p2zCj8NPuBYSOE5P"
}

# 2. SET the secret (stores as JSON string)
client.set_secret("ODSConnectionString", json.dumps(connection_info))
print("✓ Secret set successfully")

# 3. RETRIEVE the secret
retrieved_secret = client.get_secret("ODSConnectionString")
connection_string_value = retrieved_secret.value

# 4. VERIFY the format
print("\nRetrieved value:")
print(connection_string_value)

# 5. PARSE it back to verify it's valid
parsed = json.loads(connection_string_value)
print("\nParsed back to dict:")
print(parsed)
print(f"\nservertype: {parsed.get('servertype')}")

✓ Secret set successfully

Retrieved value:
{"servertype": "SQL Server", "server": "AUBOYSQL6.corp.riotinto.org", "database": "ODS_AVEVA_PROD_MANAGEMENT", "authmethod": "SQL", "username": "ODP-ETL-Databricks", "password": "p2zCj8NPuBYSOE5P"}

Parsed back to dict:
{'servertype': 'SQL Server', 'server': 'AUBOYSQL6.corp.riotinto.org', 'database': 'ODS_AVEVA_PROD_MANAGEMENT', 'authmethod': 'SQL', 'username': 'ODP-ETL-Databricks', 'password': 'p2zCj8NPuBYSOE5P'}

servertype: SQL Server
